# DAIC-WOZ Audio — v1: Honest Extraction + Baselines

## 1. Mount Drive, config & dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1a. Locate your actual Drive layout first

In [ ]:
import os, glob

PROJECT = "/content/drive/MyDrive/Depression_FL_Project/data/raw/DAIC_WOZ"

print("Searching for session folders (e.g. 300_P) under PROJECT ...")
session_hits = glob.glob(f"{PROJECT}/**/3*_P", recursive=True) + glob.glob(f"{PROJECT}/**/4*_P", recursive=True)
session_dirs = sorted(set(os.path.dirname(p) for p in session_hits))
for d in session_dirs[:5]:
    print("  candidate AUDIO_ROOT:", d)
if not session_dirs:
    print("  none found - check that the DAIC-WOZ session folders are unzipped somewhere under PROJECT")

print()
print("Searching for the label CSVs under PROJECT ...")
for fname in ["train_split_Depression_AVEC2017.csv", "dev_split_Depression_AVEC2017.csv", "full_test_split.csv"]:
    hits = glob.glob(f"{PROJECT}/**/{fname}", recursive=True)
    for h in hits:
        print(" ", fname, "->", h)
    if not hits:
        print(" ", fname, "-> NOT FOUND under PROJECT")

In [ ]:
import os

PROJECT    = "/content/drive/MyDrive/Depression_FL_Project"
AUDIO_ROOT = f"{PROJECT}/data/raw/DAIC_WOZ"   # the folder that DIRECTLY contains 300_P, 301_P, ...
OUT        = f"{PROJECT}/audio/v1"

TRAIN_CSV = f"{AUDIO_ROOT}/train_split_Depression_AVEC2017.csv"
DEV_CSV   = f"{AUDIO_ROOT}/dev_split_Depression_AVEC2017.csv"
TEST_CSV  = f"{AUDIO_ROOT}/full_test_split.csv"   # released test labels (PHQ_Binary / PHQ_Score)

os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/turn_embeddings", exist_ok=True)

# then set back to None for the full run.
DEBUG_N_SESSIONS = 5

SR            = 16000          # DAIC-WOZ audio is 16 kHz
WAVLM_MODEL   = "microsoft/wavlm-base-plus"
MAX_CHUNK_SEC = 12            # long turns are chunked to <=12 s to bound GPU memory
N_SEEDS       = 5
EXCLUDED      = {342, 394, 398, 460}   # technically excluded sessions (per documentation)

print("Project:", PROJECT)
print("Audio root:", AUDIO_ROOT, "| exists:", os.path.isdir(AUDIO_ROOT))
if os.path.isdir(AUDIO_ROOT):
    sample = [d for d in os.listdir(AUDIO_ROOT) if d.endswith("_P")][:3]
    print("  sample session folders found:", sample if sample else "NONE — check AUDIO_ROOT")
for p in (TRAIN_CSV, DEV_CSV, TEST_CSV):
    print("  label file exists:", os.path.exists(p), "->", p)

In [ ]:
!pip -q install "transformers>=4.40" torchaudio soundfile scikit-learn lightgbm 2>/dev/null
import torch, numpy as np, pandas as pd
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RNG = 42
np.random.seed(RNG); torch.manual_seed(RNG)

## 2. Load & harmonise labels

In [ ]:

def _read_labels(path, split):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    # normalise id column
    idcol = next(c for c in df.columns if c.lower() == "participant_id")
    df = df.rename(columns={idcol: "pid"})
    # binary + score columns differ across files
    bcol = "PHQ8_Binary" if "PHQ8_Binary" in df.columns else "PHQ_Binary"
    scol = "PHQ8_Score"  if "PHQ8_Score"  in df.columns else "PHQ_Score"
    out = df[["pid", bcol, scol]].copy()
    out.columns = ["pid", "y_bin", "phq8"]
    out["pid"] = out["pid"].astype(int)
    out["y_bin"] = out["y_bin"].astype(int)
    out["phq8"] = out["phq8"].astype(int)
    out["split"] = split
    return out

def severity_3class(s):
    if s <= 9:  return 0   # minimal/mild
    if s <= 14: return 1   # moderate
    return 2               # mod-severe / severe

labels = pd.concat([
    _read_labels(TRAIN_CSV, "train"),
    _read_labels(DEV_CSV,   "dev"),
    _read_labels(TEST_CSV,  "test"),
], ignore_index=True)
labels["y_sev"] = labels["phq8"].map(severity_3class)

print(labels.groupby("split").size())
print("\nbinary balance per split:")
print(labels.groupby(["split","y_bin"]).size().unstack(fill_value=0))
print("\nseverity balance per split:")
print(labels.groupby(["split","y_sev"]).size().unstack(fill_value=0))
labels.head()

## 3. Leakage guards (explicit)

In [ ]:
train_ids = set(labels.loc[labels.split=="train","pid"])
dev_ids   = set(labels.loc[labels.split=="dev",  "pid"])
test_ids  = set(labels.loc[labels.split=="test", "pid"])

assert train_ids.isdisjoint(dev_ids),  "LEAKAGE: train/dev overlap!"
assert train_ids.isdisjoint(test_ids), "LEAKAGE: train/test overlap!"
assert dev_ids.isdisjoint(test_ids),   "LEAKAGE: dev/test overlap!"
print(f"OK — disjoint subjects: train={len(train_ids)} dev={len(dev_ids)} test={len(test_ids)}")

## 4. Participant-turn intervals from transcripts

In [ ]:
import glob

def session_dir(pid):
    d = os.path.join(AUDIO_ROOT, f"{pid}_P")
    return d if os.path.isdir(d) else None

def participant_turns(pid):
    '''Return list of (start_sec, stop_sec) for clean participant speech.'''
    d = session_dir(pid)
    if d is None:
        return None
    tpath = os.path.join(d, f"{pid}_TRANSCRIPT.csv")
    if not os.path.exists(tpath):
        return None
    # transcripts are tab-separated: start_time, stop_time, speaker, value
    try:
        df = pd.read_csv(tpath, sep="\t")
    except Exception:
        df = pd.read_csv(tpath, sep="\t", engine="python", on_bad_lines="skip")
    df.columns = [c.strip().lower() for c in df.columns]
    cstart = next((c for c in df.columns if "start" in c), df.columns[0])
    cstop  = next((c for c in df.columns if "stop"  in c or "end" in c), df.columns[1])
    cspk   = next((c for c in df.columns if "speaker" in c), df.columns[2])
    cval   = next((c for c in df.columns if "value" in c or "text" in c), df.columns[-1])
    df = df[[cstart, cstop, cspk, cval]].dropna(subset=[cstart, cstop])
    df.columns = ["start", "stop", "spk", "val"]
    part = df[df["spk"].astype(str).str.lower().str.contains("participant")]
    part = part[~part["val"].astype(str).str.contains("scrubbed_entry", case=False, na=False)]
    iv = [(float(a), float(b)) for a, b in part[["start","stop"]].values if float(b) > float(a)]
    return iv

# quick sanity check on one available session
for pid in sorted(train_ids):
    iv = participant_turns(pid)
    if iv:
        print(f"pid {pid}: {len(iv)} clean turns, total {sum(b-a for a,b in iv):.1f}s")
        break

## 5. Frozen WavLM turn-level embeddings (all 13 layers)

In [ ]:
import torchaudio
from transformers import AutoFeatureExtractor, WavLMModel

feat_ext = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL)
wavlm = WavLMModel.from_pretrained(WAVLM_MODEL, output_hidden_states=True).to(DEVICE).eval()
for p in wavlm.parameters():
    p.requires_grad_(False)
    wavlm.config.mask_time_prob = 0.0          # <-- add this line
print("WavLM loaded (frozen). hidden layers:", wavlm.config.num_hidden_layers)

In [ ]:
@torch.no_grad()
def embed_turns_for_pid(pid):
    '''Return (n_turns, 13, 768) mean-pooled-per-layer turn embeddings, or None.'''
    iv = participant_turns(pid)
    d = session_dir(pid)
    if not iv or d is None:
        return None
    wpath = os.path.join(d, f"{pid}_AUDIO.wav")
    if not os.path.exists(wpath):
        return None
    wav, sr = torchaudio.load(wpath)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = wav.mean(0)  # mono
    turn_vecs = []
    max_len = MAX_CHUNK_SEC * SR
    for (a, b) in iv:
        s, e = int(a * SR), int(b * SR)
        seg = wav[s:e]
        if seg.numel() < int(0.20 * SR):   # skip <200 ms fragments
            continue
        # chunk long turns
        chunk_layer_means = []
        for c0 in range(0, seg.numel(), max_len):
            chunk = seg[c0:c0 + max_len]
            if chunk.numel() < int(0.20 * SR):
                continue
            inp = feat_ext(chunk.numpy(), sampling_rate=SR, return_tensors="pt")
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
            hs = wavlm(**inp).hidden_states            # tuple len 13, each (1, T, 768)
            layer_means = torch.stack([h.mean(1).squeeze(0) for h in hs])  # (13, 768)
            chunk_layer_means.append(layer_means.cpu())
        if chunk_layer_means:
            turn_vecs.append(torch.stack(chunk_layer_means).mean(0))       # avg chunks -> (13,768)
    if not turn_vecs:
        return None
    return torch.stack(turn_vecs).numpy()              # (n_turns, 13, 768)

all_pids = sorted(set(labels["pid"]) - EXCLUDED)
todo = [p for p in all_pids if not os.path.exists(f"{OUT}/turn_embeddings/{p}.npz")]
if DEBUG_N_SESSIONS:
    todo = todo[:DEBUG_N_SESSIONS]
    print(f"DEBUG mode: extracting {len(todo)} sessions only. Set DEBUG_N_SESSIONS=None for full run.")

import time
for i, pid in enumerate(todo, 1):
    t0 = time.time()
    emb = embed_turns_for_pid(pid)
    if emb is None:
        print(f"[{i}/{len(todo)}] pid {pid}: NO USABLE AUDIO/TURNS — skipped")
        continue
    np.savez_compressed(f"{OUT}/turn_embeddings/{pid}.npz", emb=emb)
    print(f"[{i}/{len(todo)}] pid {pid}: {emb.shape[0]} turns -> saved ({time.time()-t0:.1f}s)")
print("Extraction pass complete.")

## 6. Subject-level pooling (mean+std) per layer

In [ ]:
def pool_mean_std(turn_emb):           # (n_turns,13,768) -> (13,1536)
    mu = turn_emb.mean(0)
    sd = turn_emb.std(0)
    return np.concatenate([mu, sd], axis=-1)

pids_ok, pooled, missing = [], [], []
for pid in sorted(set(labels["pid"]) - EXCLUDED):
    f = f"{OUT}/turn_embeddings/{pid}.npz"
    if not os.path.exists(f):
        missing.append(pid); continue
    te = np.load(f)["emb"]
    pooled.append(pool_mean_std(te))
    pids_ok.append(pid)

pooled = np.stack(pooled) if pooled else np.empty((0,13,1536))
pid_index = {p: i for i, p in enumerate(pids_ok)}
print("pooled:", pooled.shape, "| subjects with audio:", len(pids_ok))
if missing:
    print(f"DATA-QUALITY NOTE: {len(missing)} subjects had no usable audio/turns and are dropped: {missing}")

def xy_for_split(split, layer, label_col="y_bin"):
    sub = labels[(labels.split == split) & (labels.pid.isin(pids_ok))]
    X = np.stack([pooled[pid_index[p], layer] for p in sub.pid])
    y = sub[label_col].values
    return X, y, sub.pid.values

In [ ]:
import random as _random, glob
import torchaudio
from torch.utils.data import Dataset, DataLoader

RUN_AUGMENT  = True
N_AUG_COPIES = 2
AUG_OUT      = f"{OUT}/turn_embeddings_aug"
os.makedirs(AUG_OUT, exist_ok=True)

class DepressionAudioAugmentation:
    def __init__(self, sample_rate=16000):
        self.sr = sample_rate
    def __call__(self, waveform_np):
        import librosa
        aug = waveform_np.astype(np.float32).copy()
        if _random.random() < 0.5:
            aug = librosa.effects.time_stretch(aug, rate=_random.uniform(0.9, 1.1))
        if _random.random() < 0.5:
            aug = librosa.effects.pitch_shift(aug, sr=self.sr, n_steps=_random.randint(-2, 2))
        if _random.random() < 0.3:
            snr_db = _random.uniform(15, 30)
            sig_power = np.mean(aug ** 2) + 1e-12
            noise_power = sig_power / (10 ** (snr_db / 10))
            aug = aug + np.random.normal(0, np.sqrt(noise_power), size=aug.shape).astype(np.float32)
        return aug

@torch.no_grad()
def embed_turns_augmented(pid, augmenter):
    iv = participant_turns(pid); d = session_dir(pid)
    if not iv or d is None:
        return None
    wpath = os.path.join(d, f"{pid}_AUDIO.wav")
    if not os.path.exists(wpath):
        return None
    wav, sr = torchaudio.load(wpath)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = torch.tensor(augmenter(wav.mean(0).numpy()))
    turn_vecs, max_len = [], MAX_CHUNK_SEC * SR
    for (a, b) in iv:
        seg = wav[int(a * SR):int(b * SR)]
        if seg.numel() < int(0.20 * SR):
            continue
        chunk_layer_means = []
        for c0 in range(0, seg.numel(), max_len):
            chunk = seg[c0:c0 + max_len]
            if chunk.numel() < int(0.20 * SR):
                continue
            inp = feat_ext(chunk.numpy(), sampling_rate=SR, return_tensors="pt")
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
            hs = wavlm(**inp).hidden_states
            chunk_layer_means.append(torch.stack([h.mean(1).squeeze(0) for h in hs]).cpu())
        if chunk_layer_means:
            turn_vecs.append(torch.stack(chunk_layer_means).mean(0))
    return torch.stack(turn_vecs).numpy() if turn_vecs else None

if RUN_AUGMENT:
    _aug = DepressionAudioAugmentation(SR)
    _train_pids_aug = [p for p in labels[labels.split == "train"].pid if p in pids_ok]
    for pid in _train_pids_aug:
        for k in range(N_AUG_COPIES):
            f = f"{AUG_OUT}/{pid}_aug{k}.npz"
            if os.path.exists(f):
                continue
            emb = embed_turns_augmented(pid, _aug)
            if emb is not None:
                np.savez_compressed(f, emb=emb)
    print(f"Augmented cache ready under {AUG_OUT} "
          f"({len(_train_pids_aug)} train subjects x up to {N_AUG_COPIES} copies).")

class TurnSetAug(Dataset):
    '''Single-layer (LOCK_LAYER) per-turn embeddings. When use_aug=True, also yields the cached
    augmented copies for each pid -- use ONLY for training pids/folds (never dev/test).'''
    def __init__(self, pids, label_col, use_aug=False):
        self.items = []
        for p in pids:
            f = f"{OUT}/turn_embeddings/{p}.npz"
            if not os.path.exists(f):
                continue
            y = int(labels.loc[labels.pid == p, label_col].iloc[0])
            te = np.load(f)["emb"][:, LOCK_LAYER, :]
            self.items.append((torch.tensor(te, dtype=torch.float32), y, p))
            if use_aug:
                for af in sorted(glob.glob(f"{AUG_OUT}/{p}_aug*.npz")):
                    tea = np.load(af)["emb"][:, LOCK_LAYER, :]
                    self.items.append((torch.tensor(tea, dtype=torch.float32), y, p))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

## 7. Layer sweep + probes (CV-driven decision rule)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import warnings; warnings.filterwarnings("ignore")

LABEL_COL = "y_bin"          # primary task; switch to "y_sev" for the secondary 3-class run
N_LAYERS  = pooled.shape[1] if pooled.size else 13

def make_probes(seed, n_classes):
    cw = "balanced"
    return {
        "logreg": LogisticRegression(max_iter=2000, class_weight=cw, C=1.0, random_state=seed),
        "svm":    SVC(kernel="linear", class_weight=cw, C=1.0, random_state=seed),
        "rf":     RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample",
                                         random_state=seed, n_jobs=-1),
        "mlp":    MLPClassifier(hidden_layer_sizes=(128,), max_iter=1000, early_stopping=True,
                                random_state=seed),
    }

def eval_layer_probe(layer, label_col=LABEL_COL):
    Xtr, ytr, _ = xy_for_split("train", layer, label_col)
    Xdv, ydv, _ = xy_for_split("dev",   layer, label_col)
    Xte, yte, _ = xy_for_split("test",  layer, label_col)
    sc = StandardScaler().fit(Xtr)
    Xtr, Xdv, Xte = sc.transform(Xtr), sc.transform(Xdv), sc.transform(Xte)
    nc = len(np.unique(ytr))
    rows = {}
    for name in ["logreg","svm","rf","mlp"]:
        dvs, tes = [], []
        for seed in range(N_SEEDS):
            clf = make_probes(seed, nc)[name].fit(Xtr, ytr)
            dvs.append(f1_score(ydv, clf.predict(Xdv), average="macro"))
            tes.append(f1_score(yte, clf.predict(Xte), average="macro"))
        rows[name] = dict(dev_mean=np.mean(dvs), dev_std=np.std(dvs),
                          test_mean=np.mean(tes), test_std=np.std(tes))
    return rows

def cv_macro_f1(layer, probe, label_col=LABEL_COL, n_splits=5):
    '''Subject-level stratified k-fold CV on pooled train+dev. No leakage: split is by subject.'''
    sub = labels[(labels.split.isin(["train","dev"])) & (labels.pid.isin(pids_ok))]
    X = np.stack([pooled[pid_index[p], layer] for p in sub.pid])
    y = sub[label_col].values
    scores = []
    for seed in range(N_SEEDS):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, va in skf.split(X, y):
            sc = StandardScaler().fit(X[tr])
            clf = make_probes(seed, len(np.unique(y)))[probe].fit(sc.transform(X[tr]), y[tr])
            scores.append(f1_score(y[va], clf.predict(sc.transform(X[va])), average="macro"))
    return np.mean(scores), np.std(scores)

results = []
for L in range(N_LAYERS):
    devtest = eval_layer_probe(L)
    for name, r in devtest.items():
        cv_m, cv_s = cv_macro_f1(L, name)
        results.append(dict(layer=L, probe=name, cv_mean=cv_m, cv_std=cv_s, **r))
    print(f"  layer {L}/{N_LAYERS-1} done")
res = pd.DataFrame(results).sort_values("cv_mean", ascending=False)
print()
print(res.head(10).to_string(index=False))

In [ ]:
best = res.iloc[0]
band = res[res.cv_mean >= best.cv_mean - best.cv_std]
locked = band.sort_values(["cv_mean","cv_std"], ascending=[False, True]).iloc[0]
print("LOCKED -> layer={}  probe={}".format(int(locked.layer), locked.probe))
print("  CV   macro-F1: {:.3f} ± {:.3f}   <- selection criterion".format(locked.cv_mean, locked.cv_std))
print("  dev  macro-F1: {:.3f} ± {:.3f}   (reported, not used to select)".format(locked.dev_mean, locked.dev_std))
print("  test macro-F1: {:.3f} ± {:.3f}   (reported, not used to select)".format(locked.test_mean, locked.test_std))
if locked.test_mean >= 0.80 or locked.cv_mean >= 0.80:
    print("  *** WARNING: macro-F1 >= 0.80 — audit the split for leakage before trusting this. ***")
LOCK_LAYER, LOCK_PROBE = int(locked.layer), locked.probe

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

LOCK_LAYER  # from §7

class TurnSet(Dataset):
    def __init__(self, pids, label_col):
        self.items = []
        for p in pids:
            f = f"{OUT}/turn_embeddings/{p}.npz"
            if not os.path.exists(f): continue
            te = np.load(f)["emb"][:, LOCK_LAYER, :]      # (n_turns, 768)
            y = int(labels.loc[labels.pid==p, label_col].iloc[0])
            self.items.append((torch.tensor(te, dtype=torch.float32), y, p))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    seqs, ys, pids = zip(*batch)
    L = max(s.shape[0] for s in seqs)
    X = torch.zeros(len(seqs), L, seqs[0].shape[1])
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i,s in enumerate(seqs):
        X[i,:s.shape[0]] = s; mask[i,:s.shape[0]] = True
    return X, mask, torch.tensor(ys), list(pids)

class AttnMLP(nn.Module):
    def __init__(self, d_in=768, d_emb=128, n_cls=2):
        super().__init__()
        self.attn = nn.Linear(d_in, 1)
        self.proj = nn.Sequential(nn.Linear(d_in, 256), nn.ReLU(), nn.Dropout(0.3),
                                  nn.Linear(256, d_emb), nn.ReLU())   # penultimate = d_emb (128)
        self.head = nn.Linear(d_emb, n_cls)
    def forward(self, x, mask):
        a = self.attn(x).squeeze(-1).masked_fill(~mask, float("-inf"))
        a = torch.softmax(a, 1).unsqueeze(-1)
        pooled = (x * a).sum(1)
        emb = self.proj(pooled)
        return self.head(emb), emb

## 8. Confirm the locked CV result

In [ ]:
m, s = cv_macro_f1(LOCK_LAYER, LOCK_PROBE)
print("Subject-level 5-fold CV (train+dev), locked layer/probe — recomputed for confirmation:")
print("  macro-F1: {:.3f} ± {:.3f}".format(m, s))

## 8b. Technique comparison

In [ ]:
from sklearn.decomposition import PCA

t1_mean, t1_std = m, s   # computed above for (LOCK_LAYER, LOCK_PROBE)

def cv_macro_f1_fused(probe, label_col=LABEL_COL, n_splits=5, n_components=64):
    sub = labels[(labels.split.isin(["train","dev"])) & (labels.pid.isin(pids_ok))]
    X_full = np.stack([pooled[pid_index[p]].reshape(-1) for p in sub.pid])   # (n, 13*1536)
    y = sub[label_col].values
    scores = []
    for seed in range(N_SEEDS):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, va in skf.split(X_full, y):
            sc = StandardScaler().fit(X_full[tr])
            Xtr_s, Xva_s = sc.transform(X_full[tr]), sc.transform(X_full[va])
            ncomp = min(n_components, len(tr) - 1)
            pca = PCA(n_components=ncomp, random_state=seed).fit(Xtr_s)
            Xtr_p, Xva_p = pca.transform(Xtr_s), pca.transform(Xva_s)
            clf = make_probes(seed, len(np.unique(y)))[probe].fit(Xtr_p, y[tr])
            scores.append(f1_score(y[va], clf.predict(Xva_p), average="macro"))
    return np.mean(scores), np.std(scores)

t2_results = {}
for probe in ["logreg", "svm", "rf"]:
    t2_results[probe] = cv_macro_f1_fused(probe)
    print(f"T2 multi-layer fusion [{probe}]: CV macro-F1 {t2_results[probe][0]:.3f} ± {t2_results[probe][1]:.3f}")
t2_best_probe = max(t2_results, key=lambda p: t2_results[p][0])
t2_mean, t2_std = t2_results[t2_best_probe]
print(f"\nT2 best probe: {t2_best_probe} -> CV macro-F1 {t2_mean:.3f} ± {t2_std:.3f}")

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class TurnSetMT(Dataset):
    '''Per-subject turn embeddings + binary label + PHQ-8 score (for the auxiliary head).'''
    def __init__(self, pids, layer=LOCK_LAYER):
        self.items = []
        for p in pids:
            f = f"{OUT}/turn_embeddings/{p}.npz"
            if not os.path.exists(f): continue
            te = np.load(f)["emb"][:, layer, :]
            row = labels.loc[labels.pid == p].iloc[0]
            self.items.append((torch.tensor(te, dtype=torch.float32),
                               int(row.y_bin), float(row.phq8), p))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate_mt(batch):
    seqs, ybin, yphq, pids = zip(*batch)
    L = max(s.shape[0] for s in seqs)
    X = torch.zeros(len(seqs), L, seqs[0].shape[1])
    mask = torch.zeros(len(seqs), L, dtype=torch.bool)
    for i, s in enumerate(seqs):
        X[i,:s.shape[0]] = s; mask[i,:s.shape[0]] = True
    return X, mask, torch.tensor(ybin), torch.tensor(yphq, dtype=torch.float32), list(pids)

class AttnMultiTask(nn.Module):
    '''Attention-pool encoder -> shared 128-dim trunk -> classification head + PHQ-8 regression head.'''
    def __init__(self, d_in=768, d_emb=128, n_cls=2):
        super().__init__()
        self.attn = nn.Linear(d_in, 1)
        self.proj = nn.Sequential(nn.Linear(d_in, 256), nn.ReLU(), nn.Dropout(0.3),
                                  nn.Linear(256, d_emb), nn.ReLU())
        self.cls_head = nn.Linear(d_emb, n_cls)
        self.reg_head = nn.Linear(d_emb, 1)     # PHQ-8 score regression (auxiliary)
    def forward(self, x, mask):
        a = self.attn(x).squeeze(-1).masked_fill(~mask, float("-inf"))
        a = torch.softmax(a, 1).unsqueeze(-1)
        pooled_x = (x * a).sum(1)
        emb = self.proj(pooled_x)
        return self.cls_head(emb), self.reg_head(emb).squeeze(-1), emb

def cv_macro_f1_multitask(reg_weight=0.3, epochs=25, lr=1e-3, n_splits=5):
    sub = labels[(labels.split.isin(["train","dev"])) & (labels.pid.isin(pids_ok))]
    all_pids_cv = sub.pid.values
    y_for_split = sub.y_bin.values
    scores = []
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in skf.split(all_pids_cv, y_for_split):
            tr_pids, va_pids = all_pids_cv[tr_idx], all_pids_cv[va_idx]
            tr_dl = DataLoader(TurnSetMT(tr_pids), batch_size=16, shuffle=True, collate_fn=collate_mt)
            net = AttnMultiTask(d_emb=128, n_cls=2).to(DEVICE)
            opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
            cls_loss_f = nn.CrossEntropyLoss(weight=torch.tensor(
                [1.0, (y_for_split==0).sum()/max(1,(y_for_split==1).sum())], dtype=torch.float32).to(DEVICE))
            reg_loss_f = nn.MSELoss()
            net.train()
            for ep in range(epochs):
                for X, mask, yb, yp, _ in tr_dl:
                    X, mask, yb, yp = X.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE), yp.to(DEVICE)
                    opt.zero_grad()
                    out_cls, out_reg, _ = net(X, mask)
                    loss = cls_loss_f(out_cls, yb) + reg_weight * reg_loss_f(out_reg, yp)
                    loss.backward(); opt.step()
            va_dl = DataLoader(TurnSetMT(va_pids), batch_size=16, collate_fn=collate_mt)
            net.eval(); P, Y = [], []
            with torch.no_grad():
                for X, mask, yb, yp, _ in va_dl:
                    out_cls, _, _ = net(X.to(DEVICE), mask.to(DEVICE))
                    P += out_cls.argmax(1).cpu().tolist(); Y += yb.tolist()
            scores.append(f1_score(Y, P, average="macro"))
    return np.mean(scores), np.std(scores)

t3_mean, t3_std = cv_macro_f1_multitask()
print(f"T3 attention-pool + multi-task MLP: CV macro-F1 {t3_mean:.3f} ± {t3_std:.3f}")

### T2b — Supervised learned layer-fusion (the real HAREN-style mechanism)

In [ ]:
class LayerFusionClassifier(nn.Module):
    '''Learnable softmax weights over the 13 WavLM layers, fused then linearly classified — trained
    end-to-end so the weighting is supervised by the depression label, not by unsupervised PCA.'''
    def __init__(self, n_layers=13, d_in=1536, n_cls=2):
        super().__init__()
        self.layer_logits = nn.Parameter(torch.zeros(n_layers))
        self.classifier = nn.Linear(d_in, n_cls)
    def forward(self, x):                      # x: (batch, n_layers, d_in)
        w = torch.softmax(self.layer_logits, dim=0)         # (n_layers,)
        fused = (x * w.view(1, -1, 1)).sum(1)                # (batch, d_in)
        return self.classifier(fused), w

def cv_macro_f1_layerfusion(label_col=LABEL_COL, n_splits=5, epochs=60, lr=1e-2):
    sub = labels[(labels.split.isin(["train","dev"])) & (labels.pid.isin(pids_ok))]
    X = np.stack([pooled[pid_index[p]] for p in sub.pid])    # (n, 13, 1536)
    y = sub[label_col].values
    scores = []
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, va in skf.split(X, y):
            sc = StandardScaler().fit(X[tr].reshape(len(tr), -1))
            Xtr = sc.transform(X[tr].reshape(len(tr), -1)).reshape(len(tr), X.shape[1], X.shape[2])
            Xva = sc.transform(X[va].reshape(len(va), -1)).reshape(len(va), X.shape[1], X.shape[2])
            Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(DEVICE)
            ytr_t = torch.tensor(y[tr], dtype=torch.long).to(DEVICE)
            Xva_t = torch.tensor(Xva, dtype=torch.float32).to(DEVICE)
            net = LayerFusionClassifier(n_layers=X.shape[1], d_in=X.shape[2], n_cls=len(np.unique(y))).to(DEVICE)
            opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
            pos = max(1, (y[tr]==1).sum()); neg = max(1, (y[tr]==0).sum())
            cls_w = torch.tensor([1.0, neg/pos], dtype=torch.float32).to(DEVICE)
            lossf = nn.CrossEntropyLoss(weight=cls_w)
            net.train()
            for ep in range(epochs):
                opt.zero_grad()
                out, _ = net(Xtr_t)
                loss = lossf(out, ytr_t)
                loss.backward(); opt.step()
            net.eval()
            with torch.no_grad():
                out, _ = net(Xva_t)
                pred = out.argmax(1).cpu().numpy()
            scores.append(f1_score(y[va], pred, average="macro"))
    return np.mean(scores), np.std(scores)

t2b_mean, t2b_std = cv_macro_f1_layerfusion()
print(f"T2b supervised layer-fusion: CV macro-F1 {t2b_mean:.3f} ± {t2b_std:.3f}")

### T4 — Hierarchical Cross-Attention Fusion (HCAF)

In [ ]:
class HierarchicalCrossAttentionFusion(nn.Module):
    '''Deep (semantic) layers attend over shallow (acoustic) layers; mid layers added as residual.
    Operates on already-pooled (13, 1536) mean+std subject vectors -- no re-extraction needed.'''
    def __init__(self, d_in=1536, d_model=256, n_heads=4, n_cls=2, dropout=0.2):
        super().__init__()
        self.shallow_proj = nn.Linear(d_in * 5, d_model)   # layers 0-4
        self.mid_proj     = nn.Linear(d_in * 3, d_model)   # layers 5-7
        self.deep_proj    = nn.Linear(d_in * 5, d_model)   # layers 8-12
        self.cross_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                                dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_cls)

    def forward(self, x):                          # x: (batch, 13, 1536)
        shallow = x[:, 0:5, :].reshape(x.shape[0], -1)
        mid     = x[:, 5:8, :].reshape(x.shape[0], -1)
        deep    = x[:, 8:13, :].reshape(x.shape[0], -1)
        s = self.shallow_proj(shallow).unsqueeze(1)   # (B,1,d_model) -> keys/values
        m = self.mid_proj(mid).unsqueeze(1)           # (B,1,d_model) -> residual
        d = self.deep_proj(deep).unsqueeze(1)         # (B,1,d_model) -> query
        attn_out, _ = self.cross_attn(query=d, key=s, value=s)
        fused = self.norm1(attn_out + m)
        out = self.norm2(fused + self.dropout(self.ffn(fused)))
        emb = out.squeeze(1)                          # (B, d_model)
        return self.classifier(emb), emb

def cv_macro_f1_hcaf(label_col=LABEL_COL, n_splits=5, epochs=80, lr=5e-4,
                     d_model=256, n_heads=4, feature_noise_std=0.05):
    sub = labels[(labels.split.isin(["train","dev"])) & (labels.pid.isin(pids_ok))]
    X = np.stack([pooled[pid_index[p]] for p in sub.pid])      # (n, 13, 1536)
    y = sub[label_col].values
    scores = []
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr, va in skf.split(X, y):
            sc = StandardScaler().fit(X[tr].reshape(len(tr), -1))
            Xtr = sc.transform(X[tr].reshape(len(tr), -1)).reshape(len(tr), X.shape[1], X.shape[2])
            Xva = sc.transform(X[va].reshape(len(va), -1)).reshape(len(va), X.shape[1], X.shape[2])
            Xtr_t = torch.tensor(Xtr, dtype=torch.float32).to(DEVICE)
            ytr_t = torch.tensor(y[tr], dtype=torch.long).to(DEVICE)
            Xva_t = torch.tensor(Xva, dtype=torch.float32).to(DEVICE)
            net = HierarchicalCrossAttentionFusion(d_in=X.shape[2], d_model=d_model, n_heads=n_heads,
                                                    n_cls=len(np.unique(y))).to(DEVICE)
            opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
            pos = max(1, (y[tr]==1).sum()); neg = max(1, (y[tr]==0).sum())
            cls_w = torch.tensor([1.0, neg/pos], dtype=torch.float32).to(DEVICE)
            lossf = nn.CrossEntropyLoss(weight=cls_w)
            net.train()
            for ep in range(epochs):
                opt.zero_grad()
                noise = torch.randn_like(Xtr_t) * feature_noise_std if feature_noise_std > 0 else 0
                out, _ = net(Xtr_t + noise)
                loss = lossf(out, ytr_t)
                loss.backward(); opt.step()
            net.eval()
            with torch.no_grad():
                out, _ = net(Xva_t)
                pred = out.argmax(1).cpu().numpy()
            scores.append(f1_score(y[va], pred, average="macro"))
    return np.mean(scores), np.std(scores)

t4_mean, t4_std = cv_macro_f1_hcaf()
print(f"T4 hierarchical cross-attention fusion (HCAF): CV macro-F1 {t4_mean:.3f} ± {t4_std:.3f}")

### T5 — Turn-Level Temporal Transformer

In [ ]:
class TurnTemporalTransformer(nn.Module):
    '''Positional encoding + small Transformer encoder over turn sequence + learned-query attention
    pool. Single-task (binary classification only) -- isolates temporal modeling from T3's multi-task
    confound.'''
    def __init__(self, d_in=768, d_model=128, n_heads=4, n_layers=2, n_cls=2, dropout=0.2, max_len=500):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                               dim_feedforward=d_model * 4, dropout=dropout,
                                               batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)   # learned pooling query
        self.pool_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                               dropout=dropout, batch_first=True)
        self.classifier = nn.Linear(d_model, n_cls)

    def forward(self, x, key_padding_mask):       # x: (B, T, d_in); mask: (B, T) True=valid
        h = self.in_proj(x) + self.pos_emb[:, :x.shape[1], :]
        pad_mask = ~key_padding_mask                                # Transformer wants True=pad
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        q = self.query.expand(x.shape[0], -1, -1)
        emb, _ = self.pool_attn(query=q, key=h, value=h, key_padding_mask=pad_mask)
        emb = emb.squeeze(1)                                         # (B, d_model)
        return self.classifier(emb), emb

def cv_macro_f1_temporal(label_col=LABEL_COL, n_splits=5, epochs=30, lr=1e-3,
                         d_model=128, n_heads=4, n_layers=2):
    sub = labels[(labels.split.isin(["train","dev"])) & (labels.pid.isin(pids_ok))]
    all_pids_cv = sub.pid.values
    y_for_split = sub[label_col].values
    scores = []
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in skf.split(all_pids_cv, y_for_split):
            tr_pids, va_pids = all_pids_cv[tr_idx], all_pids_cv[va_idx]
            tr_dl = DataLoader(TurnSetAug(tr_pids, label_col, use_aug=True), batch_size=16, shuffle=True, collate_fn=collate)
            net = TurnTemporalTransformer(d_model=d_model, n_heads=n_heads, n_layers=n_layers,
                                          n_cls=len(np.unique(y_for_split))).to(DEVICE)
            opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
            pos = max(1, (y_for_split[tr_idx]==1).sum()); neg = max(1, (y_for_split[tr_idx]==0).sum())
            cls_w = torch.tensor([1.0, neg/pos], dtype=torch.float32).to(DEVICE)
            lossf = nn.CrossEntropyLoss(weight=cls_w)
            net.train()
            for ep in range(epochs):
                for X, mask, y, _ in tr_dl:
                    X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                    opt.zero_grad(); out, _ = net(X, mask); loss = lossf(out, y); loss.backward(); opt.step()
            va_dl = DataLoader(TurnSet(va_pids, label_col), batch_size=16, collate_fn=collate)
            net.eval(); P, Y = [], []
            with torch.no_grad():
                for X, mask, y, _ in va_dl:
                    out, _ = net(X.to(DEVICE), mask.to(DEVICE))
                    P += out.argmax(1).cpu().tolist(); Y += y.tolist()
            scores.append(f1_score(Y, P, average="macro"))
    return np.mean(scores), np.std(scores)

t5_mean, t5_std = cv_macro_f1_temporal()
print(f"T5 turn-level temporal Transformer: CV macro-F1 {t5_mean:.3f} ± {t5_std:.3f}")

### T6 — CTC auxiliary loss (temporal regularization, report §5.4)

In [ ]:
import torch.nn.functional as F

class TurnCTCModel(nn.Module):
    def __init__(self, d_in=768, d_model=128, n_heads=4, n_layers=1, n_cls=2,
                 dropout=0.2, max_len=500):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                               dim_feedforward=d_model * 4, dropout=dropout,
                                               batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        # CTC head: per-turn logits over {blank=0, depressed=1}
        self.ctc_head = nn.Linear(d_model, 2)
        # classification head: learned-query attention pool
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pool_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                               dropout=dropout, batch_first=True)
        self.classifier = nn.Linear(d_model, n_cls)

    def forward(self, x, key_padding_mask):       # x:(B,T,d_in); mask:(B,T) True=valid
        h = self.in_proj(x) + self.pos_emb[:, :x.shape[1], :]
        pad_mask = ~key_padding_mask                                # True = pad
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        ctc_logits = self.ctc_head(h)                              # (B,T,2)
        q = self.query.expand(x.shape[0], -1, -1)
        emb, _ = self.pool_attn(query=q, key=h, value=h, key_padding_mask=pad_mask)
        emb = emb.squeeze(1)                                       # (B,d_model)
        return self.classifier(emb), ctc_logits, emb


def _ctc_loss_from(ctc_logits, mask, y):
    '''Weak per-turn CTC: target=[1] for depressed, empty for non-depressed.'''
    log_probs = F.log_softmax(ctc_logits, dim=-1).permute(1, 0, 2)  # (T,B,C) for CTCLoss
    input_lengths = mask.sum(1).clamp(min=1).cpu()
    tgt_list = [1 for yi in y.tolist() if yi == 1]                  # depressed token, batch order
    targets = torch.tensor(tgt_list, dtype=torch.long)
    target_lengths = torch.tensor([1 if yi == 1 else 0 for yi in y.tolist()], dtype=torch.long)
    return F.ctc_loss(log_probs, targets, input_lengths, target_lengths,
                      blank=0, zero_infinity=True)


def cv_macro_f1_ctc(label_col=LABEL_COL, n_splits=5, epochs=30, lr=1e-3,
                    d_model=128, n_heads=4, n_layers=1, ctc_weight=0.3):
    sub = labels[(labels.split.isin(["train", "dev"])) & (labels.pid.isin(pids_ok))]
    all_pids_cv = sub.pid.values
    y_for_split = sub[label_col].values
    scores = []
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in skf.split(all_pids_cv, y_for_split):
            tr_pids, va_pids = all_pids_cv[tr_idx], all_pids_cv[va_idx]
            tr_dl = DataLoader(TurnSetAug(tr_pids, label_col, use_aug=True), batch_size=16, shuffle=True,
                               collate_fn=collate)
            net = TurnCTCModel(d_model=d_model, n_heads=n_heads, n_layers=n_layers,
                               n_cls=len(np.unique(y_for_split))).to(DEVICE)
            opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
            pos = max(1, (y_for_split[tr_idx] == 1).sum()); neg = max(1, (y_for_split[tr_idx] == 0).sum())
            cls_w = torch.tensor([1.0, neg / pos], dtype=torch.float32).to(DEVICE)
            lossf = nn.CrossEntropyLoss(weight=cls_w)
            net.train()
            for ep in range(epochs):
                for X, mask, y, _ in tr_dl:
                    X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                    opt.zero_grad()
                    out, ctc_logits, _ = net(X, mask)
                    loss = lossf(out, y) + ctc_weight * _ctc_loss_from(ctc_logits, mask, y).to(DEVICE)
                    loss.backward(); opt.step()
            va_dl = DataLoader(TurnSet(va_pids, label_col), batch_size=16, collate_fn=collate)
            net.eval(); P, Y = [], []
            with torch.no_grad():
                for X, mask, y, _ in va_dl:
                    out, _, _ = net(X.to(DEVICE), mask.to(DEVICE))
                    P += out.argmax(1).cpu().tolist(); Y += y.tolist()
            scores.append(f1_score(Y, P, average="macro"))
    return np.mean(scores), np.std(scores)

t6_mean, t6_std = cv_macro_f1_ctc()
print(f"T6 turn-level CTC auxiliary loss: CV macro-F1 {t6_mean:.3f} ± {t6_std:.3f}")


In [ ]:
comparison = pd.DataFrame([
    dict(technique="T1: single-layer baseline",        cv_mean=t1_mean, cv_std=t1_std),
    dict(technique=f"T2: multi-layer fusion ({t2_best_probe})", cv_mean=t2_mean, cv_std=t2_std),
    dict(technique="T2b: supervised layer-fusion",      cv_mean=t2b_mean, cv_std=t2b_std),
    dict(technique="T3: attention-pool + multi-task",   cv_mean=t3_mean, cv_std=t3_std),
    dict(technique="T4: hierarchical cross-attention (HCAF)", cv_mean=t4_mean, cv_std=t4_std),
    dict(technique="T5: turn-level temporal Transformer", cv_mean=t5_mean, cv_std=t5_std),
    dict(technique="T6: turn-level CTC auxiliary loss",   cv_mean=t6_mean, cv_std=t6_std),
]).sort_values("cv_mean", ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))
print()
best_t = comparison.iloc[0]
band_t = comparison[comparison.cv_mean >= best_t.cv_mean - best_t.cv_std]
winner = band_t.sort_values(["cv_mean","cv_std"], ascending=[False, True]).iloc[0]
print(f"WINNER -> {winner.technique}   CV macro-F1 {winner.cv_mean:.3f} ± {winner.cv_std:.3f}")
if winner.cv_mean >= 0.75:
    print("  *** WARNING: CV macro-F1 >= 0.75 is above the literature's leakage-controlled ceiling (~0.74) — audit before trusting. ***")

WINNING_TECHNIQUE = winner.technique
print("\nThis winner determines which representation is exported as the 128-dim fusion embedding in §10.")

## 8c. Advanced components (active, per the report)

In [ ]:
# that earlier cell and re-run from there.
print('Augmentation active: cache in', AUG_OUT, '| folded into T5/T6/export training.')


In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install -q -U torchao

In [ ]:
RUN_LORA = True

if RUN_LORA:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "peft"], check=False)
    from peft import LoraConfig, get_peft_model
    from torch.utils.data import Dataset, DataLoader

    lora_config = LoraConfig(
        r=8, lora_alpha=16,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
        lora_dropout=0.1, bias="none", modules_to_save=[],
    )

    class RawTurnSet(Dataset):
        '''Yields (per-turn waveform chunks, binary label) for end-to-end fine-tuning.
        Re-reads audio from disk -- intentionally heavier than the frozen-feature datasets above.'''
        def __init__(self, pids, label_col=LABEL_COL, max_turns=40):
            self.items = []
            for p in pids:
                iv = participant_turns(p); d = session_dir(p)
                if not iv or d is None:
                    continue
                wpath = os.path.join(d, f"{p}_AUDIO.wav")
                if not os.path.exists(wpath):
                    continue
                y = int(labels.loc[labels.pid == p, label_col].iloc[0])
                self.items.append((wpath, iv[:max_turns], y))
        def __len__(self):
            return len(self.items)
        def __getitem__(self, i):
            wpath, iv, y = self.items[i]
            wav, sr = torchaudio.load(wpath)
            if sr != SR:
                wav = torchaudio.functional.resample(wav, sr, SR)
            wav = wav.mean(0)
            segs = []
            for (a, b) in iv:
                seg = wav[int(a * SR):int(b * SR)][: MAX_CHUNK_SEC * SR]
                if seg.numel() >= int(0.20 * SR):
                    segs.append(seg)
            return segs, y

    class WavLMLoRAClassifier(nn.Module):
        '''LoRA-adapted WavLM -> per-turn mean-pool -> attention-pool over turns -> classifier.'''
        def __init__(self, base_wavlm, d_in=768, d_emb=128, n_cls=2):
            super().__init__()
            self.wavlm = get_peft_model(base_wavlm, lora_config)
            self.attn = nn.Linear(d_in, 1)
            self.proj = nn.Sequential(nn.Linear(d_in, 256), nn.ReLU(), nn.Dropout(0.3),
                                      nn.Linear(256, d_emb), nn.ReLU())
            self.head = nn.Linear(d_emb, n_cls)
        def forward(self, seg_list):                       # list of 1-D waveform tensors (one subject)
            turn_vecs = []
            for seg in seg_list:
                inp = feat_ext(seg.cpu().numpy(), sampling_rate=SR, return_tensors="pt")
                inp = {k: v.to(DEVICE) for k, v in inp.items()}
                last = self.wavlm(**inp).last_hidden_state  # (1,T,768), grads flow through LoRA
                turn_vecs.append(last.mean(1).squeeze(0))
            x = torch.stack(turn_vecs).unsqueeze(0)         # (1,n_turns,768)
            a = torch.softmax(self.attn(x).squeeze(-1), 1).unsqueeze(-1)
            emb = self.proj((x * a).sum(1))
            return self.head(emb), emb

    tr_ids = [p for p in labels[labels.split == "train"].pid if p in pids_ok]
    dv_ids = [p for p in labels[labels.split == "dev"].pid   if p in pids_ok]
    n_cls = labels[LABEL_COL].nunique()
    net = WavLMLoRAClassifier(wavlm, n_cls=n_cls).to(DEVICE)
    pos = max(1, (labels.loc[labels.pid.isin(tr_ids), "y_bin"] == 1).sum())
    neg = max(1, (labels.loc[labels.pid.isin(tr_ids), "y_bin"] == 0).sum())
    lossf = nn.CrossEntropyLoss(weight=torch.tensor([1.0, neg / pos], dtype=torch.float32).to(DEVICE))
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=1e-5, weight_decay=1e-4)

    tr_ds = RawTurnSet(tr_ids)
    for ep in range(3):                                    # keep tiny: end-to-end on raw audio is slow
        net.train()
        for i in torch.randperm(len(tr_ds)).tolist():
            segs, y = tr_ds[i]
            if not segs:
                continue
            segs = [s.to(DEVICE) for s in segs]
            out, _ = net([s for s in segs])
            loss = lossf(out, torch.tensor([y], device=DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"LoRA epoch {ep + 1} done")
    print("LoRA fine-tuning finished. Evaluate on dev/test and, before reporting, wrap this in the "
          "same subject-level CV used for T1-T6 so the gain is honestly measured, not assumed.")
else:
    print("LoRA fine-tuning is defined and runnable but OFF (RUN_LORA=False). It unfreezes WavLM and "
          "trains end-to-end on raw audio -- the heaviest step here. Turn it on only after the cheap "
          "frozen-feature levers (T4/T6, augmentation) have been honestly CV-evaluated.")


## 9. Comparison track — COVAREP/FORMANT functionals

In [ ]:
from scipy import stats as sstats

DROP_HINTS = ["VUV", "creak", "HMPDM_0", "HMPDM_1", "HMPDM_2", "HMPDM_3",
              "HMPDM_4", "HMPDM_5", "HMPDM_6"]

def covarep_features(pid):
    d = session_dir(pid)
    if d is None: return None
    cpath = os.path.join(d, f"{pid}_COVAREP.csv")
    if not os.path.exists(cpath): return None
    df = pd.read_csv(cpath, header=None)   # COVAREP csv is headerless
    # column 1 is VUV (voiced flag) in the standard COVAREP DAIC layout
    vuv = df.iloc[:, 1]
    voiced = df[vuv == 1]
    if len(voiced) < 10: voiced = df
    arr = voiced.values.astype(np.float32)
    feats = np.concatenate([
        np.nanmean(arr, 0), np.nanstd(arr, 0),
        np.nanpercentile(arr, 25, 0), np.nanpercentile(arr, 75, 0),
        sstats.skew(arr, 0, nan_policy="omit"),
    ])
    return np.nan_to_num(feats)

cov, cov_pids = [], []
for pid in sorted(set(labels["pid"]) - EXCLUDED):
    f = covarep_features(pid)
    if f is not None:
        cov.append(f); cov_pids.append(pid)

if cov:
    cov = np.stack(cov); cidx = {p:i for i,p in enumerate(cov_pids)}
    def cov_xy(split, label_col=LABEL_COL):
        sub = labels[(labels.split==split) & (labels.pid.isin(cov_pids))]
        return np.stack([cov[cidx[p]] for p in sub.pid]), sub[label_col].values
    Xtr,ytr = cov_xy("train"); Xdv,ydv = cov_xy("dev"); Xte,yte = cov_xy("test")
    sc = StandardScaler().fit(Xtr)
    Xtr,Xdv,Xte = sc.transform(Xtr), sc.transform(Xdv), sc.transform(Xte)
    for name in ["logreg","svm","rf"]:
        dvs,tes=[],[]
        for seed in range(N_SEEDS):
            clf = make_probes(seed, len(np.unique(ytr)))[name].fit(Xtr,ytr)
            dvs.append(f1_score(ydv, clf.predict(Xdv), average="macro"))
            tes.append(f1_score(yte, clf.predict(Xte), average="macro"))
        print(f"COVAREP {name:6s}  dev {np.mean(dvs):.3f}±{np.std(dvs):.3f}   test {np.mean(tes):.3f}±{np.std(tes):.3f}")
else:
    print("No COVAREP CSVs found — skipping classical track.")

## 10. Export 128-dim fusion embeddings (from the §8b winner)

In [ ]:
# against the 0.56 BINARY number would be the wrong yardstick.
for _lc in ["y_bin", "y_sev"]:
    _m, _s = cv_macro_f1(LOCK_LAYER, LOCK_PROBE, label_col=_lc)
    print(f"  locked layer{LOCK_LAYER}/{LOCK_PROBE}  CV macro-F1 [{_lc}]: {_m:.3f} +/- {_s:.3f}")
print("\n^ the y_sev value is the realistic 3-class audio ceiling the FL pipeline can aim for;")
print("  the exported layer-{}-mean+std features below carry exactly this signal.".format(LOCK_LAYER))


In [ ]:
def run_mlp_final(label_col=LABEL_COL, epochs=40, lr=1e-3, multitask=False, reg_weight=0.3):
    tr_ids = [p for p in labels[labels.split=="train"].pid if p in pids_ok]
    dv_ids = [p for p in labels[labels.split=="dev"].pid   if p in pids_ok]
    n_cls = labels[label_col].nunique()
    if multitask:
        tr = DataLoader(TurnSetMT(tr_ids), batch_size=16, shuffle=True, collate_fn=collate_mt)
        pos = max(1, (labels.loc[labels.pid.isin(tr_ids), "y_bin"] == 1).sum())
        neg = max(1, (labels.loc[labels.pid.isin(tr_ids), "y_bin"] == 0).sum())
        cls_w = torch.tensor([1.0, neg / pos], dtype=torch.float32)
        reg_loss_f = nn.MSELoss()
    else:
        tr = DataLoader(TurnSetAug(tr_ids, label_col, use_aug=True), batch_size=16, shuffle=True, collate_fn=collate)
        cls_w = torch.tensor(
            [len(tr_ids) / (n_cls * max(1, (labels.loc[labels.pid.isin(tr_ids), label_col]==c).sum()))
             for c in range(n_cls)], dtype=torch.float32)
    cls_loss_f = nn.CrossEntropyLoss(weight=cls_w.to(DEVICE))
    best_dev, best_state = -1, None
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        net = (AttnMultiTask(d_emb=128, n_cls=n_cls) if multitask else AttnMLP(d_emb=128, n_cls=n_cls)).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        for ep in range(epochs):
            net.train()
            for batch in tr:
                if multitask:
                    X, mask, yb, yp, _ = batch
                    X, mask, yb, yp = X.to(DEVICE), mask.to(DEVICE), yb.to(DEVICE), yp.to(DEVICE)
                    opt.zero_grad()
                    out_cls, out_reg, _ = net(X, mask)
                    loss = cls_loss_f(out_cls, yb) + reg_weight * reg_loss_f(out_reg, yp)
                else:
                    X, mask, y, _ = batch
                    X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                    opt.zero_grad()
                    out, _ = net(X, mask)
                    loss = cls_loss_f(out, y)
                loss.backward(); opt.step()
        net.eval()
        ds = TurnSetMT(dv_ids) if multitask else TurnSet(dv_ids, label_col)
        dl = DataLoader(ds, batch_size=16, collate_fn=(collate_mt if multitask else collate))
        P, Y = [], []
        with torch.no_grad():
            for batch in dl:
                if multitask:
                    X, mask, yb, yp, _ = batch
                    out_cls, _, _ = net(X.to(DEVICE), mask.to(DEVICE)); P += out_cls.argmax(1).cpu().tolist(); Y += yb.tolist()
                else:
                    X, mask, y, _ = batch
                    out, _ = net(X.to(DEVICE), mask.to(DEVICE)); P += out.argmax(1).cpu().tolist(); Y += y.tolist()
        d = f1_score(Y, P, average="macro")
        if d > best_dev: best_dev, best_state = d, {k: v.cpu() for k, v in net.state_dict().items()}
    print(f"Final model ({'multi-task' if multitask else 'single-task'}) best-seed dev macro-F1: {best_dev:.3f}")
    return best_state, n_cls

def export_embeddings_final(state, label_col=LABEL_COL, multitask=False):
    n_cls = labels[label_col].nunique()
    net = (AttnMultiTask(d_emb=128, n_cls=n_cls) if multitask else AttnMLP(d_emb=128, n_cls=n_cls)).to(DEVICE)
    net.load_state_dict(state); net.eval()
    out = {}
    for split, tag in [("train","train"), ("dev","val"), ("test","test")]:
        ids = [p for p in labels[labels.split==split].pid if p in pids_ok]
        ds = TurnSetMT(ids) if multitask else TurnSet(ids, label_col)
        dl = DataLoader(ds, batch_size=16, collate_fn=(collate_mt if multitask else collate))
        E, Y, PIDS = [], [], []
        with torch.no_grad():
            for batch in dl:
                if multitask:
                    X, mask, yb, yp, pid = batch
                    _, _, emb = net(X.to(DEVICE), mask.to(DEVICE)); Y += yb.tolist()
                else:
                    X, mask, y, pid = batch
                    _, emb = net(X.to(DEVICE), mask.to(DEVICE)); Y += y.tolist()
                E.append(emb.cpu().numpy()); PIDS += pid
        E = np.concatenate(E) if E else np.empty((0,128))
        np.save(f"{OUT}/audio_embeddings_{tag}.npy", E)
        np.save(f"{OUT}/audio_labels_{tag}.npy", np.array(Y))
        np.save(f"{OUT}/audio_pids_{tag}.npy", np.array(PIDS))
        out[tag] = E.shape
        print(f"saved audio_embeddings_{tag}.npy -> {E.shape}")
    return out

def export_embeddings_pca128():
    '''T2 path: multi-layer concat -> PCA(128) fit on train, applied to all three splits.'''
    pca = PCA(n_components=128, random_state=0)
    out = {}
    tr_sub = labels[(labels.split=="train") & (labels.pid.isin(pids_ok))]
    Xtr_full = np.stack([pooled[pid_index[p]].reshape(-1) for p in tr_sub.pid])
    sc = StandardScaler().fit(Xtr_full)
    pca.fit(sc.transform(Xtr_full))
    for split, tag in [("train","train"), ("dev","val"), ("test","test")]:
        sub = labels[(labels.split==split) & (labels.pid.isin(pids_ok))]
        X_full = np.stack([pooled[pid_index[p]].reshape(-1) for p in sub.pid])
        E = pca.transform(sc.transform(X_full))
        np.save(f"{OUT}/audio_embeddings_{tag}.npy", E)
        np.save(f"{OUT}/audio_labels_{tag}.npy", sub.y_bin.values)
        np.save(f"{OUT}/audio_pids_{tag}.npy", sub.pid.values)
        out[tag] = E.shape
        print(f"saved audio_embeddings_{tag}.npy -> {E.shape}")
    return out

def export_embeddings_layerfusion128(label_col=LABEL_COL, epochs=80, lr=1e-2):
    '''T2b path: train LayerFusionClassifier on full train, project its 1536-dim fused
    representation to 128-dim via a learned linear bottleneck trained jointly with the classifier.'''
    tr_sub = labels[(labels.split=="train") & (labels.pid.isin(pids_ok))]
    X_tr = np.stack([pooled[pid_index[p]] for p in tr_sub.pid])
    y_tr = tr_sub[label_col].values
    sc = StandardScaler().fit(X_tr.reshape(len(X_tr), -1))

    class LayerFusionBottleneck(nn.Module):
        def __init__(self, n_layers, d_in, d_emb=128, n_cls=2):
            super().__init__()
            self.layer_logits = nn.Parameter(torch.zeros(n_layers))
            self.bottleneck = nn.Linear(d_in, d_emb)
            self.classifier = nn.Linear(d_emb, n_cls)
        def forward(self, x):
            w = torch.softmax(self.layer_logits, dim=0)
            fused = (x * w.view(1, -1, 1)).sum(1)
            emb = self.bottleneck(fused)
            return self.classifier(emb), emb

    Xtr_s = sc.transform(X_tr.reshape(len(X_tr), -1)).reshape(X_tr.shape)
    Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32).to(DEVICE)
    ytr_t = torch.tensor(y_tr, dtype=torch.long).to(DEVICE)
    pos = max(1, (y_tr==1).sum()); neg = max(1, (y_tr==0).sum())
    cls_w = torch.tensor([1.0, neg/pos], dtype=torch.float32).to(DEVICE)
    lossf = nn.CrossEntropyLoss(weight=cls_w)

    net = LayerFusionBottleneck(X_tr.shape[1], X_tr.shape[2], d_emb=128,
                                n_cls=len(np.unique(y_tr))).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
    net.train()
    for ep in range(epochs):
        opt.zero_grad(); out, _ = net(Xtr_t); loss = lossf(out, ytr_t); loss.backward(); opt.step()

    out = {}
    net.eval()
    for split, tag in [("train","train"), ("dev","val"), ("test","test")]:
        sub = labels[(labels.split==split) & (labels.pid.isin(pids_ok))]
        X = np.stack([pooled[pid_index[p]] for p in sub.pid])
        Xs = sc.transform(X.reshape(len(X), -1)).reshape(X.shape)
        with torch.no_grad():
            _, emb = net(torch.tensor(Xs, dtype=torch.float32).to(DEVICE))
        E = emb.cpu().numpy()
        np.save(f"{OUT}/audio_embeddings_{tag}.npy", E)
        np.save(f"{OUT}/audio_labels_{tag}.npy", sub[label_col].values)
        np.save(f"{OUT}/audio_pids_{tag}.npy", sub.pid.values)
        out[tag] = E.shape
        print(f"saved audio_embeddings_{tag}.npy -> {E.shape}")
    return out

def export_embeddings_hcaf128(label_col=LABEL_COL, epochs=100, lr=5e-4, feature_noise_std=0.05):
    '''T4 path: train HCAF on full train (with feature-noise regularization), project its
    256-dim fused representation to 128-dim via a learned bottleneck trained jointly.'''
    tr_sub = labels[(labels.split=="train") & (labels.pid.isin(pids_ok))]
    X_tr = np.stack([pooled[pid_index[p]] for p in tr_sub.pid])
    y_tr = tr_sub[label_col].values
    sc = StandardScaler().fit(X_tr.reshape(len(X_tr), -1))

    class HCAFBottleneck(HierarchicalCrossAttentionFusion):
        def __init__(self, d_in, d_model=256, n_heads=4, d_emb=128, n_cls=2):
            super().__init__(d_in=d_in, d_model=d_model, n_heads=n_heads, n_cls=n_cls)
            self.bottleneck = nn.Linear(d_model, d_emb)
            self.classifier = nn.Linear(d_emb, n_cls)   # overrides parent's classifier
        def forward(self, x):
            shallow = x[:, 0:5, :].reshape(x.shape[0], -1)
            mid     = x[:, 5:8, :].reshape(x.shape[0], -1)
            deep    = x[:, 8:13, :].reshape(x.shape[0], -1)
            s = self.shallow_proj(shallow).unsqueeze(1)
            m = self.mid_proj(mid).unsqueeze(1)
            d = self.deep_proj(deep).unsqueeze(1)
            attn_out, _ = self.cross_attn(query=d, key=s, value=s)
            fused = self.norm1(attn_out + m)
            out = self.norm2(fused + self.dropout(self.ffn(fused)))
            emb = self.bottleneck(out.squeeze(1))
            return self.classifier(emb), emb

    Xtr_s = sc.transform(X_tr.reshape(len(X_tr), -1)).reshape(X_tr.shape)
    Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32).to(DEVICE)
    ytr_t = torch.tensor(y_tr, dtype=torch.long).to(DEVICE)
    pos = max(1, (y_tr==1).sum()); neg = max(1, (y_tr==0).sum())
    cls_w = torch.tensor([1.0, neg/pos], dtype=torch.float32).to(DEVICE)
    lossf = nn.CrossEntropyLoss(weight=cls_w)

    net = HCAFBottleneck(X_tr.shape[2], d_emb=128, n_cls=len(np.unique(y_tr))).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-3)
    net.train()
    for ep in range(epochs):
        opt.zero_grad()
        noise = torch.randn_like(Xtr_t) * feature_noise_std if feature_noise_std > 0 else 0
        out, _ = net(Xtr_t + noise)
        loss = lossf(out, ytr_t)
        loss.backward(); opt.step()

    out = {}
    net.eval()
    for split, tag in [("train","train"), ("dev","val"), ("test","test")]:
        sub = labels[(labels.split==split) & (labels.pid.isin(pids_ok))]
        X = np.stack([pooled[pid_index[p]] for p in sub.pid])
        Xs = sc.transform(X.reshape(len(X), -1)).reshape(X.shape)
        with torch.no_grad():
            _, emb = net(torch.tensor(Xs, dtype=torch.float32).to(DEVICE))
        E = emb.cpu().numpy()
        np.save(f"{OUT}/audio_embeddings_{tag}.npy", E)
        np.save(f"{OUT}/audio_labels_{tag}.npy", sub[label_col].values)
        np.save(f"{OUT}/audio_pids_{tag}.npy", sub.pid.values)
        out[tag] = E.shape
        print(f"saved audio_embeddings_{tag}.npy -> {E.shape}")
    return out

def export_embeddings_temporal128(label_col=LABEL_COL, epochs=30, lr=1e-3):
    '''T5 path: train TurnTemporalTransformer (with a 128-dim d_model so the pooled-query
    output IS the 128-dim embedding -- no extra bottleneck needed) on the full train split.'''
    tr_ids = [p for p in labels[labels.split=="train"].pid if p in pids_ok]
    dv_ids = [p for p in labels[labels.split=="dev"].pid   if p in pids_ok]
    n_cls = labels[label_col].nunique()
    tr_dl = DataLoader(TurnSetAug(tr_ids, label_col, use_aug=True), batch_size=16, shuffle=True, collate_fn=collate)
    pos = max(1, (labels.loc[labels.pid.isin(tr_ids), label_col]==1).sum())
    neg = max(1, (labels.loc[labels.pid.isin(tr_ids), label_col]==0).sum())
    cls_w = torch.tensor([1.0, neg/pos], dtype=torch.float32).to(DEVICE)
    lossf = nn.CrossEntropyLoss(weight=cls_w)
    best_dev, best_state = -1, None
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        net = TurnTemporalTransformer(d_model=128, n_cls=n_cls).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        net.train()
        for ep in range(epochs):
            for X, mask, y, _ in tr_dl:
                X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                opt.zero_grad(); out, _ = net(X, mask); loss = lossf(out, y); loss.backward(); opt.step()
        net.eval()
        dv_dl = DataLoader(TurnSet(dv_ids, label_col), batch_size=16, collate_fn=collate)
        P, Y = [], []
        with torch.no_grad():
            for X, mask, y, _ in dv_dl:
                out, _ = net(X.to(DEVICE), mask.to(DEVICE)); P += out.argmax(1).cpu().tolist(); Y += y.tolist()
        d = f1_score(Y, P, average="macro")
        if d > best_dev: best_dev, best_state = d, {k: v.cpu() for k, v in net.state_dict().items()}
    print(f"Final T5 model best-seed dev macro-F1: {best_dev:.3f}")
    net = TurnTemporalTransformer(d_model=128, n_cls=n_cls).to(DEVICE)
    net.load_state_dict(best_state); net.eval()
    out = {}
    for split, tag in [("train","train"), ("dev","val"), ("test","test")]:
        ids = [p for p in labels[labels.split==split].pid if p in pids_ok]
        dl = DataLoader(TurnSet(ids, label_col), batch_size=16, collate_fn=collate)
        E, Y, PIDS = [], [], []
        with torch.no_grad():
            for X, mask, y, pid in dl:
                _, emb = net(X.to(DEVICE), mask.to(DEVICE)); E.append(emb.cpu().numpy()); Y += y.tolist(); PIDS += pid
        E = np.concatenate(E) if E else np.empty((0,128))
        np.save(f"{OUT}/audio_embeddings_{tag}.npy", E)
        np.save(f"{OUT}/audio_labels_{tag}.npy", np.array(Y))
        np.save(f"{OUT}/audio_pids_{tag}.npy", np.array(PIDS))
        out[tag] = E.shape
        print(f"saved audio_embeddings_{tag}.npy -> {E.shape}")
    return out

def export_embeddings_ctc128(label_col=LABEL_COL, epochs=30, lr=1e-3, ctc_weight=0.3):
    '''T6 path: train TurnCTCModel (CE + CTC) on the full train split; the 128-dim attention-pool
    output IS the exported embedding (d_model=128, no extra bottleneck), mirroring T5's export.'''
    tr_ids = [p for p in labels[labels.split == "train"].pid if p in pids_ok]
    dv_ids = [p for p in labels[labels.split == "dev"].pid   if p in pids_ok]
    n_cls = labels[label_col].nunique()
    tr_dl = DataLoader(TurnSetAug(tr_ids, label_col, use_aug=True), batch_size=16, shuffle=True, collate_fn=collate)
    pos = max(1, (labels.loc[labels.pid.isin(tr_ids), label_col] == 1).sum())
    neg = max(1, (labels.loc[labels.pid.isin(tr_ids), label_col] == 0).sum())
    cls_w = torch.tensor([1.0, neg / pos], dtype=torch.float32).to(DEVICE)
    lossf = nn.CrossEntropyLoss(weight=cls_w)
    best_dev, best_state = -1, None
    for seed in range(N_SEEDS):
        torch.manual_seed(seed)
        net = TurnCTCModel(d_model=128, n_cls=n_cls).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        net.train()
        for ep in range(epochs):
            for X, mask, y, _ in tr_dl:
                X, mask, y = X.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
                opt.zero_grad()
                out, ctc_logits, _ = net(X, mask)
                loss = lossf(out, y) + ctc_weight * _ctc_loss_from(ctc_logits, mask, y).to(DEVICE)
                loss.backward(); opt.step()
        net.eval()
        dv_dl = DataLoader(TurnSet(dv_ids, label_col), batch_size=16, collate_fn=collate)
        P, Y = [], []
        with torch.no_grad():
            for X, mask, y, _ in dv_dl:
                out, _, _ = net(X.to(DEVICE), mask.to(DEVICE))
                P += out.argmax(1).cpu().tolist(); Y += y.tolist()
        d = f1_score(Y, P, average="macro")
        if d > best_dev:
            best_dev, best_state = d, {k: v.cpu() for k, v in net.state_dict().items()}
    print(f"Final T6 (CTC) model best-seed dev macro-F1: {best_dev:.3f}")
    net = TurnCTCModel(d_model=128, n_cls=n_cls).to(DEVICE)
    net.load_state_dict(best_state); net.eval()
    out = {}
    for split, tag in [("train", "train"), ("dev", "val"), ("test", "test")]:
        ids = [p for p in labels[labels.split == split].pid if p in pids_ok]
        dl = DataLoader(TurnSet(ids, label_col), batch_size=16, collate_fn=collate)
        E, Y, PIDS = [], [], []
        with torch.no_grad():
            for X, mask, y, pid in dl:
                _, _, emb = net(X.to(DEVICE), mask.to(DEVICE))
                E.append(emb.cpu().numpy()); Y += y.tolist(); PIDS += pid
        E = np.concatenate(E) if E else np.empty((0, 128))
        np.save(f"{OUT}/audio_embeddings_{tag}.npy", E)
        np.save(f"{OUT}/audio_labels_{tag}.npy", np.array(Y))
        np.save(f"{OUT}/audio_pids_{tag}.npy", np.array(PIDS))
        out[tag] = E.shape
        print(f"saved audio_embeddings_{tag}.npy -> {E.shape}")
    return out


EXPORT_LABEL_COL = "y_sev"   # FL fusion is 3-class severity; use "y_bin" for the binary task

def export_embeddings_locked_layer(layer=None, label_col=EXPORT_LABEL_COL):
    layer = LOCK_LAYER if layer is None else layer
    tr_sub = labels[(labels.split=="train") & (labels.pid.isin(pids_ok))]
    Xtr = np.stack([pooled[pid_index[p], layer] for p in tr_sub.pid])
    sc = StandardScaler().fit(Xtr)
    out = {}
    for split, tag in [("train","train"), ("dev","val"), ("test","test")]:
        sub = labels[(labels.split==split) & (labels.pid.isin(pids_ok))]
        X = np.stack([pooled[pid_index[p], layer] for p in sub.pid])
        E = sc.transform(X).astype(np.float32)
        np.save(f"{OUT}/audio_embeddings_{tag}.npy", E)
        np.save(f"{OUT}/audio_labels_{tag}.npy",     sub[label_col].values)       # 3-class (FL)
        np.save(f"{OUT}/audio_labels_bin_{tag}.npy", sub["y_bin"].values)         # binary (kept)
        np.save(f"{OUT}/audio_pids_{tag}.npy",       sub.pid.values)
        out[tag] = E.shape
        print(f"saved audio_embeddings_{tag}.npy -> {E.shape}  (layer {layer}, label='{label_col}')")
    print(f"\nExported the LOCKED layer-{layer} mean+std representation (dim={Xtr.shape[1]}),")
    print("standardized on TRAIN only. This preserves the §7 probe signal rather than a 128-d")
    print("MLP trunk. NOTE: §7 locked on y_bin; FL uses y_sev (3-class) -- re-run the §7 sweep")
    print("with LABEL_COL='y_sev' if you need the 3-class ceiling for that representation.")
    return out


if WINNING_TECHNIQUE.startswith("T6"):
    print("Winner was T6 (CTC) -> training final CTC model on full train split.")
    export_embeddings_ctc128()
elif WINNING_TECHNIQUE.startswith("T5"):
    print("Winner was T5 (temporal Transformer) -> training final model on full train split.")
    export_embeddings_temporal128()
elif WINNING_TECHNIQUE.startswith("T4"):
    print("Winner was T4 (HCAF) -> training final HCAF+bottleneck model on full train split.")
    export_embeddings_hcaf128()
elif WINNING_TECHNIQUE.startswith("T2b"):
    print("Winner was T2b (supervised layer-fusion) -> training final fusion+bottleneck model on full train split.")
    export_embeddings_layerfusion128()
elif WINNING_TECHNIQUE.startswith("T2"):
    print("Winner was T2 (multi-layer concat+PCA) -> exporting PCA(128) embeddings.")
    export_embeddings_pca128()
elif WINNING_TECHNIQUE.startswith("T3"):
    print("Winner was T3 (attention + multi-task) -> training final multi-task model on full train split.")
    best_state, n_cls = run_mlp_final(LABEL_COL, multitask=True)
    export_embeddings_final(best_state, LABEL_COL, multitask=True)
else:
    print("Winner was T1 (single-layer baseline) -> exporting the LOCKED layer's pooled features")
    print("(faithful to the winning probe; preserves §7 signal instead of a lossy 128-d MLP trunk).")
    export_embeddings_locked_layer()

print("\nDone. 128-dim embeddings written to", OUT)

## 10b. Figures for the thesis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300, "font.size": 11})
FIGDIR = f"{OUT}/figures"
os.makedirs(FIGDIR, exist_ok=True)
print("Figures will be saved to:", FIGDIR)

In [ ]:
# Fig 1 — Layer x probe heatmap (CV macro-F1), from the §7 sweep
pivot = res.pivot_table(index="probe", columns="layer", values="cv_mean")
fig, ax = plt.subplots(figsize=(10, 3.2))
im = ax.imshow(pivot.values, aspect="auto", cmap="viridis", vmin=pivot.values.min(), vmax=pivot.values.max())
ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index)
ax.set_xlabel("WavLM layer"); ax.set_ylabel("Probe")
ax.set_title("Subject-level 5-fold CV macro-F1 by WavLM layer and probe")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v < (pivot.values.max()+pivot.values.min())/2 else "black", fontsize=7)
fig.colorbar(im, ax=ax, label="CV macro-F1", fraction=0.025, pad=0.02)
ax.scatter([LOCK_LAYER], [list(pivot.index).index(LOCK_PROBE)], marker="*", s=300,
           edgecolor="red", facecolor="none", linewidth=2, label="locked")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.15), frameon=False)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig1_layer_probe_heatmap.png", bbox_inches="tight")
plt.show()
print("saved fig1_layer_probe_heatmap.png")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
order = comparison.sort_values("cv_mean", ascending=True)
colors = ["#2ca02c" if t == WINNING_TECHNIQUE else "#4c72b0" for t in order["technique"]]
ax.barh(order["technique"], order["cv_mean"], xerr=order["cv_std"], color=colors,
        capsize=4, edgecolor="black", linewidth=0.6)
ax.axvline(0.74, color="red", linestyle="--", linewidth=1.2,
           label="Honest literature ceiling (~0.74, Yeh et al. 2026)")
ax.axvline(0.50, color="gray", linestyle=":", linewidth=1, label="Chance-level region")
ax.set_xlabel("Subject-level 5-fold CV macro-F1 (mean ± std)")
ax.set_title("Audio-only technique comparison (DAIC-WOZ, honest CV protocol)")
ax.legend(loc="lower right", fontsize=8, frameon=True)
ax.set_xlim(0, 0.85)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig2_technique_comparison.png", bbox_inches="tight")
plt.show()
print("saved fig2_technique_comparison.png")

In [ ]:
# Fig 3 — Class balance across splits (train/dev/test)
fig, ax = plt.subplots(figsize=(6, 4))
bal = labels.groupby(["split","y_bin"]).size().unstack(fill_value=0)
bal = bal.reindex(["train","dev","test"])
bal.columns = ["Non-depressed (PHQ8<10)", "Depressed (PHQ8>=10)"]
bal.plot(kind="bar", stacked=True, ax=ax, color=["#4c72b0", "#dd8452"], edgecolor="black", linewidth=0.5)
for i, (idx, row) in enumerate(bal.iterrows()):
    total = row.sum()
    ax.text(i, total + 1, f"n={int(total)}", ha="center", fontsize=9)
ax.set_xlabel(""); ax.set_ylabel("Number of subjects")
ax.set_title("Class distribution across official AVEC2017 splits")
ax.legend(loc="upper right", fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig3_class_balance.png", bbox_inches="tight")
plt.show()
print("saved fig3_class_balance.png")

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, ConfusionMatrixDisplay

Xtr, ytr, _ = xy_for_split("train", LOCK_LAYER, LABEL_COL)
Xte, yte, _ = xy_for_split("test",  LOCK_LAYER, LABEL_COL)
sc = StandardScaler().fit(Xtr)
clf = make_probes(0, len(np.unique(ytr)))[LOCK_PROBE].fit(sc.transform(Xtr), ytr)
if hasattr(clf, "decision_function"):
    scores_te = clf.decision_function(sc.transform(Xte))
else:
    scores_te = clf.predict_proba(sc.transform(Xte))[:, 1]
pred_te = clf.predict(sc.transform(Xte))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cm = confusion_matrix(yte, pred_te)
ConfusionMatrixDisplay(cm, display_labels=["Non-dep.", "Depressed"]).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Confusion matrix (test, n={len(yte)})\nlayer {LOCK_LAYER}, {LOCK_PROBE}")

fpr, tpr, _ = roc_curve(yte, scores_te); roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="#4c72b0", lw=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0,1],[0,1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC curve (test)"); axes[1].legend(loc="lower right", fontsize=9)

prec, rec, _ = precision_recall_curve(yte, scores_te); pr_auc = auc(rec, prec)
axes[2].plot(rec, prec, color="#dd8452", lw=2, label=f"AUC = {pr_auc:.3f}")
axes[2].axhline(yte.mean(), color="gray", linestyle=":", lw=1, label=f"Baseline ({yte.mean():.2f})")
axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall curve (test)"); axes[2].legend(loc="lower left", fontsize=9)

plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig4_test_diagnostics.png", bbox_inches="tight")
plt.show()
print("saved fig4_test_diagnostics.png")
print(f"\nTest macro-F1: {f1_score(yte, pred_te, average='macro'):.3f} | ROC-AUC: {roc_auc:.3f} | PR-AUC: {pr_auc:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
top = res.sort_values("cv_mean", ascending=False).head(15).reset_index(drop=True)
x = np.arange(len(top))
w = 0.27
ax.bar(x - w, top["dev_mean"], width=w, label="Dev (n=35)", color="#dd8452", edgecolor="black", linewidth=0.4)
ax.bar(x,     top["cv_mean"],  width=w, label="5-fold CV (train+dev)", color="#2ca02c", edgecolor="black", linewidth=0.4)
ax.bar(x + w, top["test_mean"],width=w, label="Test (n=47, held-out)", color="#4c72b0", edgecolor="black", linewidth=0.4)
ax.set_xticks(x)
ax.set_xticklabels([f"L{int(r.layer)}-{r.probe}" for _, r in top.iterrows()], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("macro-F1")
ax.set_title("Dev-split score vs. CV vs. held-out test (top 15 layer/probe combos)\nillustrates why CV, not raw dev score, drives model selection")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/fig5_dev_vs_cv_vs_test.png", bbox_inches="tight")
plt.show()
print("saved fig5_dev_vs_cv_vs_test.png")